# Exercise 2 — ApprovalGate

**ApprovalGate** is the human-in-the-loop guardrail.  It wraps an `approve_fn(action) -> bool` that decides whether to allow each operation.  For gate testing inject a lambda; in production it would prompt a human.  Exceptions inside `approve_fn` are caught and treated as rejections.

In [ ]:
import json

# ── Exercise: implement ApprovalGate ─────────────────────────────────────────

class ApprovalGate:
    """Human-in-the-loop gate with an injectable approve_fn."""

    def __init__(self, approve_fn=None):
        # TODO: store approve_fn; default to (lambda action: True) if None
        self._approve_fn = approve_fn if approve_fn is not None else (lambda action: True)

    def check(self, action):
        # TODO: call self._approve_fn(str(action)) inside a try/except
        # If it returns True: return (True, "approved")
        # If it returns False or raises: return (False, "rejected by approval gate")
        return True, "approved"


### Checks

In [ ]:
checks = 0

# 1 — default approve_fn auto-approves
try:
    gate = ApprovalGate()
    approved, reason = gate.check("run this action")
    assert approved
    checks += 1; print("✅ 1 default ApprovalGate auto-approves")
except Exception as e:
    print("❌ 1:", e)

# 2 — injected auto-reject fn blocks
try:
    gate = ApprovalGate(approve_fn=lambda action: False)
    approved, reason = gate.check("run this action")
    assert not approved
    checks += 1; print("✅ 2 injected auto-reject fn blocks the action")
except Exception as e:
    print("❌ 2:", e)

# 3 — check returns (bool, str) tuple
try:
    gate = ApprovalGate()
    result = gate.check("test")
    assert isinstance(result, tuple) and len(result) == 2
    assert isinstance(result[0], bool) and isinstance(result[1], str)
    checks += 1; print("✅ 3 check() returns (bool, str) tuple")
except Exception as e:
    print("❌ 3:", e)

# 4 — reason string explains rejection
try:
    gate = ApprovalGate(approve_fn=lambda a: False)
    approved, reason = gate.check("dangerous action")
    assert not approved and len(reason) > 0
    checks += 1; print("✅ 4 rejection reason is a non-empty string")
except Exception as e:
    print("❌ 4:", e)

# 5 — exception in approve_fn is treated as rejection
try:
    def bad_fn(action):
        raise RuntimeError("something went wrong")
    gate = ApprovalGate(approve_fn=bad_fn)
    approved, reason = gate.check("any action")
    assert not approved
    checks += 1; print("✅ 5 exception in approve_fn is caught and treated as rejection")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
